# 2.3 Measures of Dispersion

**Goal:** measure spread with the correct denominator, units, and robustness for the question.

**Prerequisites:** central tendency, squares, roots, and measurement scales. **Runtime:** Python 3.10+, standard library only. Examples are synthetic; saved outputs are included. Run cells in order.

## Contents
1. Why spread matters
2. Range
3. Interquartile range (IQR)
4. Mean absolute deviation
5. Variance, population variance, and sample variance
6. Standard deviation
7. Coefficient of variation
8. ML scaling and outlier example
9. Practice and solutions

## 1. Center is not enough

Datasets 4,5,6 and 0,5,10 share the mean 5 but differ greatly in spread. Dispersion describes variation around or across a distribution. It helps compare consistency, identify unusual observations, and choose feature scales.

Spread does not by itself explain why variation exists. Mixing distinct populations can produce a large spread even when each group is individually consistent. Define units and groups before interpreting a summary.

## 2. Range

$$R=x_{\max}-x_{\min}.$$

The range is simple and in the original units, but depends only on two observations and is very sensitive to extremes. It also tends to grow as more observations are collected, so comparisons across very different sample sizes need care.

**Worked example:** 1,2,3,4,5,6,7,8,30 have range $30-1=29$. A single value of 300 instead of 30 would increase the range to 299 without changing the other eight values.

## 3. Interquartile Range (IQR)

$$\operatorname{IQR}=Q_3-Q_1.$$

$Q_1$ and $Q_3$ are the 25th and 75th percentiles. The IQR describes the width of the middle part of the distribution and is more resistant to extreme magnitudes than the range or variance. It has the original units.

**Quantile convention:** these notebooks sort the data and linearly interpolate at zero-based position $(n-1)p$ for percentile fraction $p$. This is often called type 7. Other conventions, such as splitting the data into halves, can give different quartiles in small samples. Always state the convention rather than assuming every tool agrees.

**Worked example:** for 1,2,3,4,5,6,7,8,30, $Q_1=3$, $Q_3=7$, so IQR = 4. The upper Tukey fence is $7+1.5(4)=13$; 30 is flagged for investigation. A fence is a heuristic, not proof of an erroneous value.

In [1]:
import math
import statistics

def quantile(values, p):
    """Linear interpolation at position (n-1)*p (often called type 7)."""
    xs = sorted(values)
    if not xs or not 0 <= p <= 1:
        raise ValueError("Need nonempty values and p in [0, 1]")
    position = (len(xs)-1)*p
    low = math.floor(position)
    high = math.ceil(position)
    return xs[low] + (position-low)*(xs[high]-xs[low])

xs = [1, 2, 3, 4, 5, 6, 7, 8, 30]
q1, q3 = quantile(xs, 0.25), quantile(xs, 0.75)
iqr = q3-q1
print("Range:", max(xs)-min(xs))
print("Q1:", q1, "Q3:", q3, "IQR:", iqr)
print("1.5 IQR fences:", q1-1.5*iqr, q3+1.5*iqr)
assert (q1, q3, iqr) == (3, 7, 4)

Range: 29
Q1: 3.0 Q3: 7.0 IQR: 4.0
1.5 IQR fences: -3.0 13.0


## 4. Mean Absolute Deviation

The **mean absolute deviation about the mean** is
$$\operatorname{MeanAD}=\frac1n\sum_i|x_i-\bar x|.$$
It averages distances from the mean, avoiding cancellation of positive and negative errors, and is in original units. A different center can be chosen, but it must be named.

**Worked example:** for 2,4,6, mean 4 gives absolute deviations 2,0,2, so mean absolute deviation is $4/3\approx1.3333$.

Do not confuse this with **median absolute deviation**, $\operatorname{MedianAD}=\operatorname{median}_i|x_i-\operatorname{median}(x)|$. The abbreviation “MAD” is ambiguous; many robust-statistics tools mean the median-based quantity. Mean absolute deviation about the mean is not as resistant to extremes as the median-based version.

In [2]:
simple = [2, 4, 6]
center = statistics.mean(simple)
mean_ad = statistics.mean(abs(x-center) for x in simple)
median_center = statistics.median(simple)
median_ad = statistics.median(abs(x-median_center) for x in simple)
print("Mean absolute deviation about mean:", mean_ad)
print("Median absolute deviation about median:", median_ad)
assert math.isclose(mean_ad, 4/3)

Mean absolute deviation about mean: 1.3333333333333333
Median absolute deviation about median: 2


## 5. Variance; Population Variance; Sample Variance

Variance averages squared deviations. Squaring prevents cancellation and emphasizes large deviations, but changes the units to squared units.

For a complete finite population of size $N$:
$$\mu=\frac1N\sum_i x_i,\qquad \sigma^2=\frac1N\sum_i(x_i-\mu)^2.$$

For $n>1$ observations used to estimate the variance of an underlying population:
$$s^2=\frac1{n-1}\sum_i(x_i-\bar x)^2.$$

Once the sample mean has been estimated, deviations sum to zero, leaving $n-1$ freely varying deviations. Under independent, identically distributed sampling with finite variance, dividing by $n-1$ makes $s^2$ an unbiased estimator of the generating population variance. It does not fix selection bias, dependence, or poor measurement.

**Worked example:** for 2,4,6, squared deviations from 4 are 4,0,4, total 8. Treating the three values as the entire finite population gives variance $8/3$. Using them as an IID sample to estimate a wider population variance gives $8/2=4$.

The denominator reflects the task, not merely whether a dataset is “large.” Describing the empirical distribution of a sample can legitimately use denominator $n$; estimating a population variance is a different goal. A single observation has empirical variance 0, but the $n-1$ sample variance is undefined.

In [3]:
squared_deviations = [(x-statistics.mean(simple))**2 for x in simple]
pop_variance = statistics.pvariance(simple)
sample_variance = statistics.variance(simple)
print("Squared deviations:", squared_deviations, "total:", sum(squared_deviations))
print("Population/empirical variance:", pop_variance)
print("Sample variance estimator:", sample_variance)
assert math.isclose(pop_variance, 8/3)
assert sample_variance == 4

Squared deviations: [4, 0, 4] total: 8
Population/empirical variance: 2.6666666666666665
Sample variance estimator: 4


## 6. Standard Deviation

Standard deviation is the positive square root of variance: $\sigma=\sqrt{\sigma^2}$ or $s=\sqrt{s^2}$. It restores original measurement units. In the previous example, population SD is $\sqrt{8/3}\approx1.6330$ and sample SD is 2.

For a transformation $y=ax+b$, variance becomes $a^2\operatorname{Var}(x)$ and SD becomes $|a|\operatorname{SD}(x)$. Adding a constant shifts the center without changing spread. Multiplying seconds by 1000 to get milliseconds multiplies SD by 1000 and variance by $10^6$.

Standard deviation is not the standard error of a mean. SD describes variation in individual values; standard error describes the sampling variability of an estimator. Also, $s^2$ being unbiased does not imply that $s$ itself is an unbiased estimator of $\sigma$.

## 7. Coefficient of Variation

$$\operatorname{CV}=\frac{s}{\bar x},\qquad\operatorname{CV\%}=100\frac{s}{\bar x}.$$

For positive ratio-scale quantities with a nonzero mean, CV summarizes relative variability without units. It is invariant to a positive unit conversion that does not shift zero. It is not generally meaningful on interval scales with arbitrary zeros, such as Celsius. Near-zero means make it unstable; zero means make it undefined, and negative means complicate interpretation.

**Worked example:** process A has mean 10 and SD 2, CV 20%; process B has mean 100 and SD 5, CV 5%. B has greater absolute spread but lower relative spread. Comparisons still require compatible constructs and a relevant purpose.

In [4]:
print("Population SD:", statistics.pstdev(simple), "Sample SD:", statistics.stdev(simple))
transformed = [10*x+7 for x in simple]
print("Transformed sample SD:", statistics.stdev(transformed))
print("Process A CV (%):", 100*2/10, "Process B CV (%):", 100*5/100)
assert math.isclose(statistics.stdev(transformed), 10*statistics.stdev(simple))

Population SD: 1.632993161855452 Sample SD: 2.0
Transformed sample SD: 20.0
Process A CV (%): 20.0 Process B CV (%): 5.0


## 8. AI/ML application: robust scaling and outlier flags

An IQR-based robust transform is $(x-\operatorname{median}_{train})/\operatorname{IQR}_{train}$. A standard transform uses training mean and SD. Both require a policy for zero spread. Scaling does not remove outliers or make a distribution normal, and fitted scaling values must not use held-out data.

IQR fences can identify unusual training values for review. Do not automatically delete or cap observations simply because they cross a fence: rare high costs or fraud events may be exactly what the model must learn. Evaluate transformations in the context of the target task and the collection process.

In [5]:
train = [1, 2, 3, 4, 5, 6, 7, 8, 30]
train_median = statistics.median(train)
train_iqr = quantile(train, 0.75)-quantile(train, 0.25)
assert train_iqr > 0
robust_scaled = [(x-train_median)/train_iqr for x in train]
low = quantile(train, 0.25)-1.5*train_iqr
high = quantile(train, 0.75)+1.5*train_iqr
flags = [x for x in train if x < low or x > high]
print("Training median:", train_median, "IQR:", train_iqr)
print("Robust scaled:", robust_scaled)
print("Investigate these fence flags:", flags)
assert flags == [30]

Training median: 5 IQR: 4.0
Robust scaled: [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 6.25]
Investigate these fence flags: [30]


## Practice

1. For 2,4,6, find range, mean absolute deviation about the mean, population variance, and sample SD.
2. For sorted values 1 through 9, find $Q_1$, $Q_3$, and IQR using the stated interpolation rule.
3. If all values are shifted by +100, what happens to range, variance, and SD?
4. If values in meters are converted to centimeters, how do variance and SD change?
5. Two processes have (mean, SD) of (20,4) and (50,5). Compare their CVs.
6. Why is CV unstable when a mean is close to zero? Why is it problematic for Celsius?
7. Is sample variance with denominator $n-1$ defined for a single value?
8. Does a value above the upper IQR fence have to be a recording error?

## Solutions

1. Range 4; mean absolute deviation $4/3$; population variance $8/3$; sample SD 2.
2. Positions $8(0.25)=2$ and $8(0.75)=6$ give $Q_1=3$, $Q_3=7$, IQR 4.
3. All three are unchanged because deviations and pairwise distances are unchanged.
4. Values multiply by 100, so variance multiplies by 10000 and SD by 100.
5. 20% and 10%; the second has greater absolute SD but less relative variability.
6. A tiny denominator makes the ratio volatile or undefined. Celsius has an arbitrary zero, so changing the origin changes CV without changing physical variation.
7. No, its denominator is zero. There is not enough information for that variance estimator.
8. No. The rule is a descriptive screening heuristic; verify the measurement and assess substantive relevance.

**Readiness check:** name the dispersion statistic precisely, use the correct units and denominator, and distinguish a flag from a decision to delete data.

**Next:** [2.4 Position and Ranking](4_Position-and-Ranking.ipynb).